# KNN Pipeline For `knn_dataset.csv`

This notebook builds a simple KNN classification workflow on top of `data/knn_dataset.csv`.

It covers:
- dataset loading and validation
- grouped, stratified cross-validation using the fixture-like prefix in `scored_after_eval_key`
- a leakage-safe preprocessing + KNN pipeline
- a compact hyperparameter search ranked by mean PR AUC


In [1]:
from __future__ import annotations

import os
import warnings

os.environ['LOKY_MAX_CPU_COUNT'] = '1'
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, balanced_accuracy_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

ROOT = next(path for path in [Path.cwd(), Path.cwd().parent] if (path / 'data').exists())
DATA_DIR = ROOT / 'data'
RANDOM_STATE = 42
N_SPLITS = 5
PRIMARY_METRIC = 'pr_auc'


In [2]:
def to_integer_frame(values):
    if isinstance(values, pd.DataFrame):
        return values.astype(int)
    return pd.DataFrame(values).astype(int)


def metric_row(y_true, y_pred, y_proba):
    return {
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'roc_auc': roc_auc_score(y_true, y_proba),
        'pr_auc': average_precision_score(y_true, y_proba),
        'brier_score': brier_score_loss(y_true, y_proba),
    }


def build_preprocessor(numeric_features, categorical_features, boolean_features):
    numeric_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ])

    categorical_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ])

    boolean_transformer = Pipeline([
        ('to_int', FunctionTransformer(to_integer_frame, validate=False)),
        ('imputer', SimpleImputer(strategy='most_frequent')),
    ])

    return ColumnTransformer([
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
        ('bool', boolean_transformer, boolean_features),
    ])


def build_knn_pipeline(numeric_features, categorical_features, boolean_features, params):
    preprocessor = build_preprocessor(numeric_features, categorical_features, boolean_features)
    classifier = KNeighborsClassifier(**params)
    return Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', classifier),
    ])


def cross_validate_knn(X, y, groups, numeric_features, categorical_features, boolean_features, params):
    splitter = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    fold_rows = []

    for fold_number, (train_idx, val_idx) in enumerate(splitter.split(X, y, groups), start=1):
        X_train = X.iloc[train_idx]
        X_val = X.iloc[val_idx]
        y_train = y.iloc[train_idx]
        y_val = y.iloc[val_idx]

        model = build_knn_pipeline(
            numeric_features=numeric_features,
            categorical_features=categorical_features,
            boolean_features=boolean_features,
            params=params,
        )
        model.fit(X_train, y_train)

        y_pred = model.predict(X_val)
        y_proba = model.predict_proba(X_val)[:, 1]
        fold_metric_row = metric_row(y_true=y_val, y_pred=y_pred, y_proba=y_proba)
        fold_metric_row.update({
            'fold': fold_number,
            'rows': len(val_idx),
            'positives': int(y_val.sum()),
        })
        fold_rows.append(fold_metric_row)

    fold_metrics = pd.DataFrame(fold_rows)
    mean_metrics = fold_metrics[['balanced_accuracy', 'roc_auc', 'pr_auc', 'brier_score']].mean().to_dict()
    summary = {
        'mean_balanced_accuracy': mean_metrics['balanced_accuracy'],
        'mean_roc_auc': mean_metrics['roc_auc'],
        'mean_pr_auc': mean_metrics['pr_auc'],
        'mean_brier_score': mean_metrics['brier_score'],
    }
    summary.update(params)
    return fold_metrics, summary


In [3]:
df = pd.read_csv(DATA_DIR / 'knn_dataset.csv')

required_columns = {
    'scored_after',
    'scored_after_eval_key',
    'checkpoint',
    'position',
    'formation',
    'minute_in_window',
    'is_home',
    'subbed',
}
missing_required_columns = sorted(required_columns - set(df.columns))
if missing_required_columns:
    raise ValueError(f'Missing required columns: {missing_required_columns}')

if not df['scored_after_eval_key'].is_unique:
    raise ValueError('`scored_after_eval_key` must be unique.')

unique_target_values = set(df['scored_after'].dropna().unique())
if unique_target_values - {0, 1}:
    raise ValueError(f'`scored_after` must be binary. Found: {sorted(unique_target_values)}')

df['group_id'] = df['scored_after_eval_key'].astype(str).str.split('__').str[0]

dataset_summary = pd.DataFrame([
    {
        'rows': len(df),
        'columns': len(df.columns),
        'missing_cells': int(df.isna().sum().sum()),
        'positive_rate': df['scored_after'].mean(),
        'unique_groups': df['group_id'].nunique(),
    }
])

class_balance = (
    df['scored_after']
    .value_counts(dropna=False)
    .rename_axis('scored_after')
    .reset_index(name='rows')
)
class_balance['share'] = class_balance['rows'] / len(df)

group_summary = (
    df.groupby('group_id', as_index=False)
    .agg(rows=('group_id', 'size'), positives=('scored_after', 'sum'))
    .sort_values('group_id')
)
group_summary['positive_rate'] = group_summary['positives'] / group_summary['rows']

display(dataset_summary)
display(class_balance)
display(group_summary.head(10))


,rows,columns,missing_cells,positive_rate,unique_groups
0,3486,32,0,0.058233,31


,scored_after,rows,share
0,0,3283,0.941767
1,1,203,0.058233


,group_id,rows,positives,positive_rate
0,1159,110,13,0.118182
1,1160,110,1,0.009091
2,1161,110,12,0.109091
3,1162,110,0,0.000000
4,1163,110,3,0.027273
5,1164,110,13,0.118182
6,1165,110,5,0.045455
7,1166,110,13,0.118182
8,1167,110,5,0.045455
9,1169,110,0,0.000000


In [4]:
target_column = 'scored_after'
drop_columns = ['scored_after', 'scored_after_eval_key', 'group_id']

categorical_features = ['checkpoint', 'position', 'formation', 'minute_in_window']
boolean_features = ['is_home', 'subbed']
numeric_features = [
    column
    for column in df.columns
    if column not in drop_columns + categorical_features + boolean_features
]

feature_roles = pd.DataFrame({
    'feature': numeric_features + categorical_features + boolean_features,
    'role': (
        ['numeric'] * len(numeric_features)
        + ['categorical'] * len(categorical_features)
        + ['boolean'] * len(boolean_features)
    ),
})

X = df.drop(columns=drop_columns)
y = df[target_column].astype(int)
groups = df['group_id']

display(feature_roles)
print(f'Numeric features: {len(numeric_features)}')
print(f'Categorical features: {len(categorical_features)}')
print(f'Boolean features: {len(boolean_features)}')


,feature,role
0,player_id,numeric
1,jersey_number,numeric
2,last15_sprints,numeric
3,last15_hsr,numeric
4,last15_shots,numeric
5,last15_shots_on_target,numeric
6,last15_shots_under_press,numeric
7,last15_shots_top_third,numeric
8,cumul_sprints,numeric
9,cumul_hsr,numeric


Numeric features: 23
Categorical features: 4
Boolean features: 2


In [5]:
baseline_params = {
    'n_neighbors': 15,
    'weights': 'distance',
    'metric': 'minkowski',
    'p': 2,
}

baseline_fold_metrics, baseline_summary = cross_validate_knn(
    X=X,
    y=y,
    groups=groups,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    boolean_features=boolean_features,
    params=baseline_params,
)

baseline_summary_df = pd.DataFrame([baseline_summary])
display(baseline_fold_metrics)
display(baseline_summary_df)


,balanced_accuracy,roc_auc,pr_auc,brier_score,fold,rows,positives
0,0.5,0.652893,0.087252,0.054621,1,594,33
1,0.5,0.545868,0.102867,0.064755,2,586,39
2,0.5,0.583020,0.081106,0.056539,3,770,44
3,0.5,0.584044,0.109554,0.058658,4,770,46
4,0.5,0.534146,0.065191,0.055399,5,766,41


,mean_balanced_accuracy,mean_roc_auc,mean_pr_auc,mean_brier_score,n_neighbors,weights,metric,p
0,0.5,0.579994,0.089194,0.057994,15,distance,minkowski,2


In [6]:
search_space = []
for n_neighbors, weights, p in product([5, 11, 21, 31], ['uniform', 'distance'], [1, 2]):
    search_space.append({
        'n_neighbors': n_neighbors,
        'weights': weights,
        'metric': 'minkowski',
        'p': p,
    })

tuning_rows = []
all_fold_results = {}
for params in search_space:
    fold_metrics, summary = cross_validate_knn(
        X=X,
        y=y,
        groups=groups,
        numeric_features=numeric_features,
        categorical_features=categorical_features,
        boolean_features=boolean_features,
        params=params,
    )
    trial_key = (
        f"k={params['n_neighbors']}_"
        f"weights={params['weights']}_"
        f"p={params['p']}"
    )
    all_fold_results[trial_key] = fold_metrics
    summary['trial_key'] = trial_key
    tuning_rows.append(summary)

tuning_results = pd.DataFrame(tuning_rows).sort_values(
    ['mean_pr_auc', 'mean_roc_auc', 'mean_balanced_accuracy'],
    ascending=[False, False, False],
).reset_index(drop=True)

display(tuning_results)


,mean_balanced_accuracy,mean_roc_auc,mean_pr_auc,mean_brier_score,n_neighbors,weights,metric,p,trial_key
0,0.500000,0.615248,0.096333,0.055767,31,distance,minkowski,1,k=31_weights=distance_p=1
1,0.500000,0.606766,0.095342,0.055940,31,distance,minkowski,2,k=31_weights=distance_p=2
2,0.499862,0.571368,0.089136,0.059339,11,distance,minkowski,1,k=11_weights=distance_p=1
3,0.500000,0.605915,0.088399,0.055937,31,uniform,minkowski,2,k=31_weights=uniform_p=2
4,0.499725,0.569789,0.088267,0.059370,11,distance,minkowski,2,k=11_weights=distance_p=2
5,0.500000,0.616642,0.086933,0.055713,31,uniform,minkowski,1,k=31_weights=uniform_p=1
6,0.500000,0.595748,0.086923,0.056828,21,distance,minkowski,2,k=21_weights=distance_p=2
7,0.500000,0.588928,0.086466,0.057104,21,distance,minkowski,1,k=21_weights=distance_p=1
8,0.506039,0.541421,0.086201,0.064868,5,distance,minkowski,1,k=5_weights=distance_p=1
9,0.500000,0.594565,0.082633,0.056834,21,uniform,minkowski,2,k=21_weights=uniform_p=2


In [7]:
best_trial = tuning_results.iloc[0].to_dict()
best_trial_key = best_trial['trial_key']
best_fold_metrics = all_fold_results[best_trial_key].copy()

best_summary = pd.DataFrame([
    {
        'trial_key': best_trial['trial_key'],
        'n_neighbors': int(best_trial['n_neighbors']),
        'weights': best_trial['weights'],
        'p': int(best_trial['p']),
        'mean_pr_auc': best_trial['mean_pr_auc'],
        'mean_roc_auc': best_trial['mean_roc_auc'],
        'mean_balanced_accuracy': best_trial['mean_balanced_accuracy'],
        'mean_brier_score': best_trial['mean_brier_score'],
    }
])

print('Best configuration ranked by mean PR AUC:')
display(best_summary)
print('Best configuration fold metrics:')
display(best_fold_metrics)


Best configuration ranked by mean PR AUC:


,trial_key,n_neighbors,weights,p,mean_pr_auc,mean_roc_auc,mean_balanced_accuracy,mean_brier_score
0,k=31_weights=distance_p=1,31,distance,1,0.096333,0.615248,0.5,0.055767


Best configuration fold metrics:


,balanced_accuracy,roc_auc,pr_auc,brier_score,fold,rows,positives
0,0.5,0.712580,0.116511,0.051459,1,594,33
1,0.5,0.607228,0.096593,0.062608,2,586,39
2,0.5,0.588812,0.088304,0.054765,3,770,44
3,0.5,0.616518,0.111851,0.056995,4,770,46
4,0.5,0.551102,0.068403,0.053009,5,766,41


## Notes

- Cross-validation is both grouped and stratified to reduce leakage risk and stabilize the rare positive class across folds.
- PR AUC is the ranking metric for tuning because the target rate is low.
- The notebook stays intentionally simple: it does not export a fitted model or write artifacts outside the notebook file itself.


 The current best
  tested setup is n_neighbors=31, weights="distance", p=1, with mean PR AUC 0.0963 and
  mean ROC AUC 0.6152.

## Extension: Drop ID-like Features And Tune Thresholds

This extension keeps the original notebook workflow unchanged and adds a follow-up experiment.

Goals:
- remove `player_id` and `jersey_number` from the KNN distance calculation
- rerun the compact KNN search on the reduced feature set
- tune the classification threshold on out-of-fold probabilities instead of relying on `0.5`


In [8]:
reduced_drop_features = ['player_id', 'jersey_number']
X_reduced = X.drop(columns=reduced_drop_features).copy()
numeric_features_reduced = [
    feature for feature in numeric_features if feature not in reduced_drop_features
]

reduced_feature_roles = pd.DataFrame({
    'feature': numeric_features_reduced + categorical_features + boolean_features,
    'role': (
        ['numeric'] * len(numeric_features_reduced)
        + ['categorical'] * len(categorical_features)
        + ['boolean'] * len(boolean_features)
    ),
})

reduced_feature_summary = pd.DataFrame([
    {
        'dropped_features': ', '.join(reduced_drop_features),
        'remaining_numeric_features': len(numeric_features_reduced),
        'remaining_total_features': X_reduced.shape[1],
    }
])

display(reduced_feature_summary)
display(reduced_feature_roles)


,dropped_features,remaining_numeric_features,remaining_total_features
0,"player_id, jersey_number",21,27


,feature,role
0,last15_sprints,numeric
1,last15_hsr,numeric
2,last15_shots,numeric
3,last15_shots_on_target,numeric
4,last15_shots_under_press,numeric
5,last15_shots_top_third,numeric
6,cumul_sprints,numeric
7,cumul_hsr,numeric
8,cumul_shots,numeric
9,cumul_shots_on_target,numeric


In [9]:
reduced_tuning_rows = []
reduced_all_fold_results = {}

for params in search_space:
    fold_metrics, summary = cross_validate_knn(
        X=X_reduced,
        y=y,
        groups=groups,
        numeric_features=numeric_features_reduced,
        categorical_features=categorical_features,
        boolean_features=boolean_features,
        params=params,
    )
    trial_key = (
        f"k={params['n_neighbors']}_"
        f"weights={params['weights']}_"
        f"p={params['p']}"
    )
    reduced_all_fold_results[trial_key] = fold_metrics
    summary['trial_key'] = trial_key
    reduced_tuning_rows.append(summary)

reduced_tuning_results = pd.DataFrame(reduced_tuning_rows).sort_values(
    ['mean_pr_auc', 'mean_roc_auc', 'mean_balanced_accuracy'],
    ascending=[False, False, False],
).reset_index(drop=True)

reduced_best_trial = reduced_tuning_results.iloc[0].to_dict()
reduced_best_trial_key = reduced_best_trial['trial_key']
reduced_best_fold_metrics = reduced_all_fold_results[reduced_best_trial_key].copy()
reduced_best_summary = pd.DataFrame([
    {
        'trial_key': reduced_best_trial['trial_key'],
        'n_neighbors': int(reduced_best_trial['n_neighbors']),
        'weights': reduced_best_trial['weights'],
        'p': int(reduced_best_trial['p']),
        'mean_pr_auc': reduced_best_trial['mean_pr_auc'],
        'mean_roc_auc': reduced_best_trial['mean_roc_auc'],
        'mean_balanced_accuracy': reduced_best_trial['mean_balanced_accuracy'],
        'mean_brier_score': reduced_best_trial['mean_brier_score'],
    }
])

print('Reduced-feature tuning results:')
display(reduced_tuning_results)
print('Best reduced-feature configuration ranked by mean PR AUC:')
display(reduced_best_summary)
print('Best reduced-feature fold metrics:')
display(reduced_best_fold_metrics)


Reduced-feature tuning results:


,mean_balanced_accuracy,mean_roc_auc,mean_pr_auc,mean_brier_score,n_neighbors,weights,metric,p,trial_key
0,0.500000,0.625086,0.096700,0.055543,31,distance,minkowski,2,k=31_weights=distance_p=2
1,0.500000,0.611474,0.095223,0.056346,21,distance,minkowski,2,k=21_weights=distance_p=2
2,0.500000,0.617864,0.094411,0.055766,31,distance,minkowski,1,k=31_weights=distance_p=1
3,0.500000,0.599932,0.092957,0.056799,21,distance,minkowski,1,k=21_weights=distance_p=1
4,0.500000,0.627525,0.092661,0.055487,31,uniform,minkowski,2,k=31_weights=uniform_p=2
5,0.500000,0.582120,0.090631,0.058904,11,distance,minkowski,2,k=11_weights=distance_p=2
6,0.500000,0.620412,0.090232,0.055631,31,uniform,minkowski,1,k=31_weights=uniform_p=1
7,0.500000,0.616547,0.086053,0.056270,21,uniform,minkowski,2,k=21_weights=uniform_p=2
8,0.503102,0.540919,0.082707,0.065180,5,distance,minkowski,1,k=5_weights=distance_p=1
9,0.500000,0.603635,0.081797,0.056693,21,uniform,minkowski,1,k=21_weights=uniform_p=1


Best reduced-feature configuration ranked by mean PR AUC:


,trial_key,n_neighbors,weights,p,mean_pr_auc,mean_roc_auc,mean_balanced_accuracy,mean_brier_score
0,k=31_weights=distance_p=2,31,distance,2,0.0967,0.625086,0.5,0.055543


Best reduced-feature fold metrics:


,balanced_accuracy,roc_auc,pr_auc,brier_score,fold,rows,positives
0,0.5,0.697348,0.118594,0.051393,1,594,33
1,0.5,0.660620,0.101115,0.061973,2,586,39
2,0.5,0.584914,0.085229,0.054981,3,770,44
3,0.5,0.634894,0.118565,0.056136,4,770,46
4,0.5,0.547653,0.060000,0.053231,5,766,41


In [10]:
from sklearn.metrics import precision_score, recall_score


def collect_oof_probabilities(X_frame, y_series, group_series, numeric_feature_names, params):
    splitter = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    oof_rows = []

    for fold_number, (train_idx, val_idx) in enumerate(splitter.split(X_frame, y_series, group_series), start=1):
        model = build_knn_pipeline(
            numeric_features=numeric_feature_names,
            categorical_features=categorical_features,
            boolean_features=boolean_features,
            params=params,
        )
        model.fit(X_frame.iloc[train_idx], y_series.iloc[train_idx])
        probabilities = model.predict_proba(X_frame.iloc[val_idx])[:, 1]

        fold_frame = pd.DataFrame({
            'fold': fold_number,
            'y_true': y_series.iloc[val_idx].to_numpy(),
            'y_proba': probabilities,
        })
        oof_rows.append(fold_frame)

    return pd.concat(oof_rows, ignore_index=True)


reduced_best_params = {
    'n_neighbors': int(reduced_best_trial['n_neighbors']),
    'weights': reduced_best_trial['weights'],
    'metric': 'minkowski',
    'p': int(reduced_best_trial['p']),
}

reduced_oof = collect_oof_probabilities(
    X_frame=X_reduced,
    y_series=y,
    group_series=groups,
    numeric_feature_names=numeric_features_reduced,
    params=reduced_best_params,
)

threshold_rows = []
for threshold in np.arange(0.05, 0.31, 0.01):
    y_pred = (reduced_oof['y_proba'] >= threshold).astype(int)
    threshold_rows.append({
        'threshold': round(float(threshold), 2),
        'balanced_accuracy': balanced_accuracy_score(reduced_oof['y_true'], y_pred),
        'precision': precision_score(reduced_oof['y_true'], y_pred, zero_division=0),
        'recall': recall_score(reduced_oof['y_true'], y_pred, zero_division=0),
        'predicted_positive_rate': y_pred.mean(),
    })

threshold_results = pd.DataFrame(threshold_rows).sort_values(
    ['balanced_accuracy', 'recall', 'precision'],
    ascending=[False, False, False],
).reset_index(drop=True)

best_threshold_row = threshold_results.iloc[0].to_dict()
reduced_oof_pr_auc = average_precision_score(reduced_oof['y_true'], reduced_oof['y_proba'])
reduced_oof_roc_auc = roc_auc_score(reduced_oof['y_true'], reduced_oof['y_proba'])

threshold_summary = pd.DataFrame([
    {
        'selected_threshold': best_threshold_row['threshold'],
        'balanced_accuracy': best_threshold_row['balanced_accuracy'],
        'precision': best_threshold_row['precision'],
        'recall': best_threshold_row['recall'],
        'predicted_positive_rate': best_threshold_row['predicted_positive_rate'],
        'oof_pr_auc': reduced_oof_pr_auc,
        'oof_roc_auc': reduced_oof_roc_auc,
    }
])

print('Threshold search on reduced-feature out-of-fold probabilities:')
display(threshold_results.head(10))
print('Selected reduced-feature threshold summary:')
display(threshold_summary)


Threshold search on reduced-feature out-of-fold probabilities:


,threshold,balanced_accuracy,precision,recall,predicted_positive_rate
0,0.05,0.609808,0.081967,0.714286,0.507458
1,0.06,0.588470,0.079375,0.625616,0.458979
2,0.09,0.582016,0.089374,0.443350,0.288870
3,0.08,0.580903,0.086796,0.463054,0.310671
4,0.07,0.579816,0.084184,0.487685,0.337349
5,0.10,0.554807,0.087258,0.310345,0.207114
6,0.12,0.553657,0.093913,0.266010,0.164945
7,0.11,0.553279,0.090620,0.280788,0.180436
8,0.13,0.545753,0.098174,0.211823,0.125645
9,0.14,0.545622,0.107649,0.187192,0.101262


Selected reduced-feature threshold summary:


,selected_threshold,balanced_accuracy,precision,recall,predicted_positive_rate,oof_pr_auc,oof_roc_auc
0,0.05,0.609808,0.081967,0.714286,0.507458,0.089303,0.622328


That improved the best mean ranking metrics slightly:
  PR AUC moved from 0.0963 to 0.0967, ROC AUC from 0.6152 to 0.6251, with the best
  reduced-feature setup becoming n_neighbors=31, weights="distance", p=2.

  The more meaningful gain came from threshold tuning. With the reduced-feature model,
  the best tested threshold was 0.05, which raised balanced accuracy to 0.6098 and
  recall to 0.7143. The tradeoff is that precision stays low at 0.0820 and the model
  predicts positives for about half the rows, so this is better if you care about
  catching positives, not if you need selective alerts.

## Extension: PCA Before KNN

This extension applies PCA after the reduced-feature preprocessing stack and before KNN.

It keeps the earlier reduced-feature setup and tests whether a compact latent representation improves distance-based behavior.

In [11]:
from sklearn.decomposition import PCA


def to_float_array(values):
    return np.asarray(values, dtype=np.float64)


def build_knn_pca_pipeline(numeric_features, categorical_features, boolean_features, n_components, params):
    preprocessor = build_preprocessor(numeric_features, categorical_features, boolean_features)
    return Pipeline([
        ('preprocessor', preprocessor),
        ('to_float', FunctionTransformer(to_float_array, validate=False)),
        ('post_scale', StandardScaler()),
        ('pca', PCA(n_components=n_components, svd_solver='full', random_state=RANDOM_STATE)),
        ('classifier', KNeighborsClassifier(**params)),
    ])


def cross_validate_knn_pca(X_frame, y_series, group_series, numeric_feature_names, categorical_feature_names, boolean_feature_names, n_components, params):
    splitter = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    fold_rows = []

    for fold_number, (train_idx, val_idx) in enumerate(splitter.split(X_frame, y_series, group_series), start=1):
        model = build_knn_pca_pipeline(
            numeric_features=numeric_feature_names,
            categorical_features=categorical_feature_names,
            boolean_features=boolean_feature_names,
            n_components=n_components,
            params=params,
        )
        with warnings.catch_warnings():
            warnings.filterwarnings('ignore', category=RuntimeWarning)
            model.fit(X_frame.iloc[train_idx], y_series.iloc[train_idx])
            y_proba = model.predict_proba(X_frame.iloc[val_idx])[:, 1]

        y_val = y_series.iloc[val_idx]
        y_pred = (y_proba >= 0.5).astype(int)
        fold_metric_row = metric_row(y_true=y_val, y_pred=y_pred, y_proba=y_proba)
        fold_metric_row.update({
            'fold': fold_number,
            'rows': len(val_idx),
            'positives': int(y_val.sum()),
        })
        fold_rows.append(fold_metric_row)

    fold_metrics = pd.DataFrame(fold_rows)
    mean_metrics = fold_metrics[['balanced_accuracy', 'roc_auc', 'pr_auc', 'brier_score']].mean().to_dict()
    summary = {
        'mean_balanced_accuracy': mean_metrics['balanced_accuracy'],
        'mean_roc_auc': mean_metrics['roc_auc'],
        'mean_pr_auc': mean_metrics['pr_auc'],
        'mean_brier_score': mean_metrics['brier_score'],
        'n_components': n_components,
    }
    summary.update(params)
    return fold_metrics, summary


In [12]:
pca_component_grid = [8, 12, 16, 20, 24]
pca_tuning_rows = []
pca_all_fold_results = {}

for n_components in pca_component_grid:
    for params in search_space:
        fold_metrics, summary = cross_validate_knn_pca(
            X_frame=X_reduced,
            y_series=y,
            group_series=groups,
            numeric_feature_names=numeric_features_reduced,
            categorical_feature_names=categorical_features,
            boolean_feature_names=boolean_features,
            n_components=n_components,
            params=params,
        )
        trial_key = (
            f"components={n_components}_"
            f"k={params['n_neighbors']}_"
            f"weights={params['weights']}_"
            f"p={params['p']}"
        )
        pca_all_fold_results[trial_key] = fold_metrics
        summary['trial_key'] = trial_key
        pca_tuning_rows.append(summary)

pca_tuning_results = pd.DataFrame(pca_tuning_rows).sort_values(
    ['mean_pr_auc', 'mean_roc_auc', 'mean_balanced_accuracy'],
    ascending=[False, False, False],
).reset_index(drop=True)

pca_best_trial = pca_tuning_results.iloc[0].to_dict()
pca_best_trial_key = pca_best_trial['trial_key']
pca_best_fold_metrics = pca_all_fold_results[pca_best_trial_key].copy()
pca_best_summary = pd.DataFrame([
    {
        'trial_key': pca_best_trial['trial_key'],
        'n_components': int(pca_best_trial['n_components']),
        'n_neighbors': int(pca_best_trial['n_neighbors']),
        'weights': pca_best_trial['weights'],
        'p': int(pca_best_trial['p']),
        'mean_pr_auc': pca_best_trial['mean_pr_auc'],
        'mean_roc_auc': pca_best_trial['mean_roc_auc'],
        'mean_balanced_accuracy': pca_best_trial['mean_balanced_accuracy'],
        'mean_brier_score': pca_best_trial['mean_brier_score'],
    }
])

pca_comparison = pd.DataFrame([
    {
        'model_variant': 'reduced_knn_no_pca',
        'mean_pr_auc': reduced_best_trial['mean_pr_auc'],
        'mean_roc_auc': reduced_best_trial['mean_roc_auc'],
        'mean_balanced_accuracy': reduced_best_trial['mean_balanced_accuracy'],
        'mean_brier_score': reduced_best_trial['mean_brier_score'],
    },
    {
        'model_variant': 'reduced_knn_with_pca',
        'mean_pr_auc': pca_best_trial['mean_pr_auc'],
        'mean_roc_auc': pca_best_trial['mean_roc_auc'],
        'mean_balanced_accuracy': pca_best_trial['mean_balanced_accuracy'],
        'mean_brier_score': pca_best_trial['mean_brier_score'],
    },
])

print('PCA tuning results:')
display(pca_tuning_results.head(15))
print('Best PCA configuration ranked by mean PR AUC:')
display(pca_best_summary)
print('Best PCA configuration fold metrics:')
display(pca_best_fold_metrics)
print('Reduced-feature KNN vs PCA-extended KNN:')
display(pca_comparison)


PCA tuning results:


,mean_balanced_accuracy,mean_roc_auc,mean_pr_auc,mean_brier_score,n_components,n_neighbors,weights,metric,p,trial_key
0,0.500000,0.602592,0.107873,0.055928,12,31,distance,minkowski,1,components=12_k=31_weights=distance_p=1
1,0.500000,0.614079,0.106988,0.055939,24,31,distance,minkowski,1,components=24_k=31_weights=distance_p=1
2,0.500000,0.602638,0.104433,0.056963,24,21,distance,minkowski,1,components=24_k=21_weights=distance_p=1
3,0.500000,0.594195,0.104329,0.056231,8,31,distance,minkowski,2,components=8_k=31_weights=distance_p=2
4,0.500000,0.598818,0.103777,0.056095,12,31,distance,minkowski,2,components=12_k=31_weights=distance_p=2
5,0.500000,0.603952,0.103351,0.055656,12,31,uniform,minkowski,1,components=12_k=31_weights=uniform_p=1
6,0.500000,0.604288,0.103249,0.056154,16,31,distance,minkowski,1,components=16_k=31_weights=distance_p=1
7,0.500000,0.594115,0.101987,0.056501,24,31,distance,minkowski,2,components=24_k=31_weights=distance_p=2
8,0.499724,0.583827,0.101159,0.056979,12,21,distance,minkowski,1,components=12_k=21_weights=distance_p=1
9,0.500000,0.591989,0.101122,0.056473,20,31,distance,minkowski,2,components=20_k=31_weights=distance_p=2


Best PCA configuration ranked by mean PR AUC:


,trial_key,n_components,n_neighbors,weights,p,mean_pr_auc,mean_roc_auc,mean_balanced_accuracy,mean_brier_score
0,components=12_k=31_weights=distance_p=1,12,31,distance,1,0.107873,0.602592,0.5,0.055928


Best PCA configuration fold metrics:


,balanced_accuracy,roc_auc,pr_auc,brier_score,fold,rows,positives
0,0.5,0.770324,0.199220,0.049130,1,594,33
1,0.5,0.561806,0.087964,0.063680,2,586,39
2,0.5,0.592756,0.086236,0.054883,3,770,44
3,0.5,0.538824,0.089848,0.058847,4,770,46
4,0.5,0.549251,0.076095,0.053098,5,766,41


Reduced-feature KNN vs PCA-extended KNN:


,model_variant,mean_pr_auc,mean_roc_auc,mean_balanced_accuracy,mean_brier_score
0,reduced_knn_no_pca,0.096700,0.625086,0.5,0.055543
1,reduced_knn_with_pca,0.107873,0.602592,0.5,0.055928


In [13]:
def collect_oof_probabilities_pca(X_frame, y_series, group_series, numeric_feature_names, categorical_feature_names, boolean_feature_names, n_components, params):
    splitter = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    oof_rows = []

    for fold_number, (train_idx, val_idx) in enumerate(splitter.split(X_frame, y_series, group_series), start=1):
        model = build_knn_pca_pipeline(
            numeric_features=numeric_feature_names,
            categorical_features=categorical_feature_names,
            boolean_features=boolean_feature_names,
            n_components=n_components,
            params=params,
        )
        with warnings.catch_warnings():
            warnings.filterwarnings('ignore', category=RuntimeWarning)
            model.fit(X_frame.iloc[train_idx], y_series.iloc[train_idx])
            probabilities = model.predict_proba(X_frame.iloc[val_idx])[:, 1]

        fold_frame = pd.DataFrame({
            'fold': fold_number,
            'y_true': y_series.iloc[val_idx].to_numpy(),
            'y_proba': probabilities,
        })
        oof_rows.append(fold_frame)

    return pd.concat(oof_rows, ignore_index=True)


pca_best_params = {
    'n_neighbors': int(pca_best_trial['n_neighbors']),
    'weights': pca_best_trial['weights'],
    'metric': 'minkowski',
    'p': int(pca_best_trial['p']),
}

pca_oof = collect_oof_probabilities_pca(
    X_frame=X_reduced,
    y_series=y,
    group_series=groups,
    numeric_feature_names=numeric_features_reduced,
    categorical_feature_names=categorical_features,
    boolean_feature_names=boolean_features,
    n_components=int(pca_best_trial['n_components']),
    params=pca_best_params,
)

pca_threshold_rows = []
for threshold in np.arange(0.05, 0.31, 0.01):
    y_pred = (pca_oof['y_proba'] >= threshold).astype(int)
    pca_threshold_rows.append({
        'threshold': round(float(threshold), 2),
        'balanced_accuracy': balanced_accuracy_score(pca_oof['y_true'], y_pred),
        'precision': precision_score(pca_oof['y_true'], y_pred, zero_division=0),
        'recall': recall_score(pca_oof['y_true'], y_pred, zero_division=0),
        'predicted_positive_rate': y_pred.mean(),
    })

pca_threshold_results = pd.DataFrame(pca_threshold_rows).sort_values(
    ['balanced_accuracy', 'recall', 'precision'],
    ascending=[False, False, False],
).reset_index(drop=True)

pca_best_threshold_row = pca_threshold_results.iloc[0].to_dict()
pca_threshold_summary = pd.DataFrame([
    {
        'selected_threshold': pca_best_threshold_row['threshold'],
        'balanced_accuracy': pca_best_threshold_row['balanced_accuracy'],
        'precision': pca_best_threshold_row['precision'],
        'recall': pca_best_threshold_row['recall'],
        'predicted_positive_rate': pca_best_threshold_row['predicted_positive_rate'],
        'oof_pr_auc': average_precision_score(pca_oof['y_true'], pca_oof['y_proba']),
        'oof_roc_auc': roc_auc_score(pca_oof['y_true'], pca_oof['y_proba']),
    }
])

print('PCA threshold search on out-of-fold probabilities:')
display(pca_threshold_results.head(10))
print('Selected PCA threshold summary:')
display(pca_threshold_summary)


PCA threshold search on out-of-fold probabilities:


,threshold,balanced_accuracy,precision,recall,predicted_positive_rate
0,0.08,0.570945,0.085064,0.423645,0.290017
1,0.09,0.569160,0.089307,0.374384,0.244119
2,0.10,0.567017,0.097117,0.315271,0.189042
3,0.07,0.560085,0.078193,0.443350,0.330178
4,0.11,0.557974,0.096386,0.275862,0.166667
5,0.05,0.551073,0.069781,0.581281,0.485083
6,0.06,0.546488,0.070822,0.492611,0.405049
7,0.12,0.544587,0.093750,0.221675,0.137694
8,0.13,0.535365,0.093264,0.177340,0.110729
9,0.18,0.527913,0.132867,0.093596,0.041021


Selected PCA threshold summary:


,selected_threshold,balanced_accuracy,precision,recall,predicted_positive_rate,oof_pr_auc,oof_roc_auc
0,0.08,0.570945,0.085064,0.423645,0.290017,0.087755,0.596227
